# Data Ingestion, Summarization and Storage

This notebook walks through the full data pipeline:

1. **Fetch & chunk** — pull Wikipedia articles and split into ~250-word paragraphs
2. **Summarize** — generate three style variants per chunk via OpenRouter (shorten / professional / informal)
3. **Inspect** — quick sanity checks on the output

The output JSONL (`data/summaries_v3.jsonl`) feeds the training pipeline in `02_lora_laplace_training.ipynb`.

All heavy-lifting is done by `src.data_pipeline` — this notebook only sets parameters and calls that module.

## 1. Configuration

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR       = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
TITLES_FILE    = DATA_DIR / "wikipedia.lst"
CHUNKS_FILE    = DATA_DIR / "wikipedia_chunks.jsonl"
SUMMARIES_FILE = DATA_DIR / "summaries_v3.jsonl"

# ── Chunking ──────────────────────────────────────────────────────────────────
APPROX_WORDS = 250

# ── OpenRouter ────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY  = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL    = "openai/gpt-4o-mini"
SUMMARY_TEMPERATURE = 0.7
SUMMARY_MAX_TOKENS  = 200
WORKERS             = 4
N_MAX               = None   # set to an int to limit chunks processed

assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY before running this notebook."
print("Config OK")

## 2. Define Wikipedia titles

In [ ]:
TITLES = [
    "Clocks",
    "Machine learning",
    "Climate change",
    "Vaccination",
    "Quantum computing",
    "Photosynthesis",
    "History of the Internet",
    "Black hole",
    "CRISPR",
    "Ancient Rome",
]

TITLES_FILE.write_text("\n".join(TITLES) + "\n")
print(f"Wrote {len(TITLES)} titles to {TITLES_FILE}")

## 3. Fetch and chunk Wikipedia articles

In [ ]:
from tqdm.notebook import tqdm
from src.data_pipeline import build_session, fetch_and_chunk_titles, write_jsonl
from src.nltk_setup import ensure_sentence_tokenizer

ensure_sentence_tokenizer(download=True)

titles  = [t.strip() for t in TITLES_FILE.read_text().splitlines() if t.strip()]
session = build_session()
written = 0

CHUNKS_FILE.unlink(missing_ok=True)
for chunk in tqdm(fetch_and_chunk_titles(titles, approx_words=APPROX_WORDS, session=session),
                  desc="Fetching & chunking"):
    write_jsonl(str(CHUNKS_FILE), chunk)
    written += 1

print(f"Wrote {written} chunks to {CHUNKS_FILE}")

## 4. Generate summaries via OpenRouter

In [ ]:
import json
from src.data_pipeline import read_jsonl, summarize_chunks, SUMMARY_STYLES

chunks = list(read_jsonl(str(CHUNKS_FILE)))
if N_MAX is not None:
    chunks = chunks[:N_MAX]
print(f"Summarizing {len(chunks)} chunks × {len(SUMMARY_STYLES)} styles = {len(chunks)*len(SUMMARY_STYLES)} API calls")

SUMMARIES_FILE.unlink(missing_ok=True)
written = errors = 0

for result in tqdm(
    summarize_chunks(chunks, model=OPENROUTER_MODEL, temperature=SUMMARY_TEMPERATURE,
                     max_tokens=SUMMARY_MAX_TOKENS, api_key=OPENROUTER_API_KEY, workers=WORKERS),
    total=len(chunks) * len(SUMMARY_STYLES), desc="Summarizing",
):
    if "error" in result:
        print(f"[ERROR] {result}")
        errors += 1
    else:
        write_jsonl(str(SUMMARIES_FILE), result)
        written += 1

print(f"Wrote {written} summaries, {errors} errors -> {SUMMARIES_FILE}")

## 5. Inspect results

In [ ]:
import pandas as pd

records = list(read_jsonl(str(SUMMARIES_FILE)))
df = pd.DataFrame(records)

print(f"Total records : {len(df)}")
print(f"Styles        : {df['summary_style'].value_counts().to_dict()}")
print(f"Pages         : {df['page_title'].nunique()}")
df["src_words"] = df["paragraph_text"].str.split().str.len()
df["tgt_words"] = df["summary"].str.split().str.len()
print("\nSource length (words):")
print(df["src_words"].describe().round(1))
print("\nSummary length (words):")
print(df["tgt_words"].describe().round(1))

In [ ]:
# Show one example per style for the first chunk
first_base = df["id"].str.rsplit("|", n=1).str[0].iloc[0]
for _, row in df[df["id"].str.startswith(first_base)].iterrows():
    print(f"=== {row['summary_style'].upper()} ===")
    print(f"SOURCE : {row['paragraph_text'][:200]}...")
    print(f"SUMMARY: {row['summary']}")
    print()